# Membership Churn Analysis

## Notebook 02: Data Cleaning & Transformation

---

**Author:** D. 
**Date:** February 25, 2026 
**Dataset:** churn_t_db.csv (2.1GB, 18.4M rows)

---


### Dependencies

In [0]:
# ═══════════════════════════════════════════════════════════════
# Dependencies
# ═══════════════════════════════════════════════════════════════

import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, StringType, DoubleType
import pandas as pd

print("Dependencies loaded successfully")

Dependencies loaded successfully


## 1.0 Setup & Configuration

### 1.1 Environment Verification

**CONTEXT**

To establish the foundational environment for analysing the Organisation's membership churn dataset. This includes verifying Spark availability, confirming file access, and validating the uploaded CSV before any transformations begin.

**PURPOSE**

To ensure:
1. PySpark environment is properly initialised
2. The uploaded CSV file is accessible at the expected location
3. File size matches expectations (~2.1GB)
4. Confident progression to data loading

**STEP**

Confirm that `churn_t_db.csv` exists at the expected Unity Catalog volume path and compute its size in gigabytes. Verify PySpark environment initialisation.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 1.1 Environment Verification
# ═══════════════════════════════════════════════════════════════

# Check Spark version
print(f"Spark Version: {spark.version}")
print("-" * 70)

# Define file path
file_path = "/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv"

# Verify file exists and get details
file_info = dbutils.fs.ls("/Volumes/workspace/rcn_churn/raw_data/")

# Display file information
display(file_info)

# Calculate and print file size
for file in file_info:
    if "churn_t_db" in file.name:
        size_gb = file.size / (1024**3)
        print(f"\nFile: {file.name}")
        print(f"Size: {size_gb:.2f} GB")

Spark Version: 4.1.0
----------------------------------------------------------------------


path,name,size,modificationTime
dbfs:/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv,churn_t_db.csv,2216363321,1771178702000
dbfs:/Volumes/workspace/rcn_churn/raw_data/delta_raw/,delta_raw/,0,1772409801749



File: churn_t_db.csv
Size: 2.06 GB


**RESULT**

Spark 4.1.0 is initialised and operational. `churn_t_db.csv` was located successfully at `/Volumes/workspace/rcn_churn/raw_data/churn_t_db.csv` and measures 2.06 GB, consistent with the expected ~2.1 GB. The Bronze table from Notebook 01 is also confirmed present at `delta_raw/`. Environment is validated and ready for data loading.

**Status:** ✓ Pass

### 1.2 Data Loading

**CONTEXT**

Notebook 01 ingested the raw CSV and persisted it as the Bronze table at `/Volumes/workspace/rcn_churn/raw_data/delta_raw/`. Notebook 02 loads directly from this table rather than re-reading the CSV, preserving the medallion architecture and ensuring all downstream transformations operate on a consistent, versioned data source.

**PURPOSE**

To ensure:
1. The Bronze table loads successfully with the expected row and column count
2. The schema from Notebook 01 is preserved and consistent
3. A verified baseline exists before any cleaning transformations are applied
4. Confident progression to data cleaning and transformation

**STEP**

Load the Bronze table into a PySpark DataFrame and confirm row count, column count, and schema against the baseline established in Notebook 01.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 1.2 Data Loading
# ═══════════════════════════════════════════════════════════════

# Define Bronze table path
bronze_path = "/Volumes/workspace/rcn_churn/raw_data/delta_raw/"

# Load Bronze table
df = spark.read.format("delta").load(bronze_path)

# Confirm row and column count
row_count = df.count()
col_count = len(df.columns)

print(f"Rows    : {row_count:,}")
print(f"Columns : {col_count}")
print("-" * 70)

# Display schema
df.printSchema()

Rows    : 18,461,480
Columns : 12
----------------------------------------------------------------------
root
 |-- _c0: integer (nullable = true)
 |-- CM_snapshot_date: date (nullable = true)
 |-- Int_nurse: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- MemCategory: string (nullable = true)
 |-- CatName: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- YoB: double (nullable = true)
 |-- MemSectorType: string (nullable = true)
 |-- YoJ: double (nullable = true)
 |-- q_members_t: double (nullable = true)
 |-- q_leavers_t: double (nullable = true)



**RESULT**

The Bronze table loaded successfully with 18,461,480 rows and 12 columns, consistent with the baseline established in Notebook 01. The schema is preserved as expected, with `YoB`, `YoJ`, `q_members_t`, and `q_leavers_t` stored as `double` type, corrections for `YoB` and `YoJ` will be addressed in Section 2.0.

**Status:** ✓ Pass

### 1.3 Configuration & Constants


**CONTEXT**

To centralise all notebook configuration in a single location, establishing path constants, business rules, and validation thresholds that will be referenced throughout the cleaning and transformation pipeline.

**PURPOSE**

To ensure:
1. All file paths are defined consistently and referenced from a single source
2. Business rules for data validation are explicitly documented
3. Cleaning thresholds are transparent and reproducible
4. Any future changes to configuration require a single update point

**STEP**

Define all path constants, business rule thresholds, and configuration variables required for Notebook 02's cleaning and transformation pipeline.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 1.3 Configuration & Constants
# ═══════════════════════════════════════════════════════════════

# ── Paths ──────────────────────────────────────────────────────
BRONZE_PATH = "/Volumes/workspace/rcn_churn/raw_data/delta_raw/"
SILVER_PATH = "/Volumes/workspace/rcn_churn/silver/churn_cleaned/"

# ── Business Rules ─────────────────────────────────────────────
YOB_MIN      = 1895   # earliest valid year of birth
YOB_MAX      = 2007   # latest valid year of birth (minimum age ~14)
YOJ_MIN      = 1941   # earliest valid year of join
YOJ_MAX      = 2025   # latest valid year of join
MIN_JOIN_AGE = 14     # minimum plausible membership join age
MAX_JOIN_AGE = 90     # maximum plausible membership join age

# ── Run Metadata ───────────────────────────────────────────────
RUN_TS       = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print("Configuration loaded successfully")
print("-" * 70)
print(f"Bronze Path  : {BRONZE_PATH}")
print(f"Silver Path  : {SILVER_PATH}")
print(f"YoB Range    : {YOB_MIN} - {YOB_MAX}")
print(f"YoJ Range    : {YOJ_MIN} - {YOJ_MAX}")
print(f"Join Age     : {MIN_JOIN_AGE} - {MAX_JOIN_AGE}")
print(f"Run Time     : {RUN_TS}")

Configuration loaded successfully
----------------------------------------------------------------------
Bronze Path  : /Volumes/workspace/rcn_churn/raw_data/delta_raw/
Silver Path  : /Volumes/workspace/rcn_churn/silver/churn_cleaned/
YoB Range    : 1895 - 2007
YoJ Range    : 1941 - 2025
Join Age     : 14 - 90
Run Time     : 2026-03-02 00:03:23


**RESULT**

All configuration constants loaded successfully. File paths, business rule thresholds, and run metadata are centralised and ready to be referenced throughout the cleaning and transformation pipeline.

**Status:** ✓ Pass

## 2.0 Data Cleaning & Transformation

### 2.1 Data Type Corrections

**CONTEXT**

Schema inference during CSV ingestion in Notebook 01 stored `YoB` and `YoJ` as `double` type, as Spark interpreted year values such as `1985.0` as floating point numbers. Both columns represent discrete calendar years and should be integer type. Additionally, `q_members_t` and `q_leavers_t` were stored as `double`, which is appropriate for weighted membership units and requires no correction.

**PURPOSE**

To ensure:
1. `YoB` and `YoJ` are stored as integer type, eliminating floating point representation
2. All downstream age and tenure calculations operate on integers
3. Schema accurately reflects the nature of each column
4. No data loss occurs during type conversion

**STEP**

Cast `YoB` and `YoJ` from `double` to `IntegerType` and confirm the schema update. Verify no values were distorted during the cast by sampling before and after.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.1 Data Type Corrections
# ═══════════════════════════════════════════════════════════════

# Cast YoB and YoJ from double to integer
df = df.withColumn("YoB", F.col("YoB").cast(IntegerType())) \
       .withColumn("YoJ", F.col("YoJ").cast(IntegerType()))

# Confirm updated schema
print("Updated Schema:")
print("-" * 70)
df.printSchema()

# Sample check to confirm no value distortion
print("Sample YoB and YoJ values (post-cast):")
print("-" * 70)
df.select("YoB", "YoJ").dropna().show(5, truncate=False)

Updated Schema:
----------------------------------------------------------------------
root
 |-- _c0: integer (nullable = true)
 |-- CM_snapshot_date: date (nullable = true)
 |-- Int_nurse: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- MemCategory: string (nullable = true)
 |-- CatName: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- YoB: integer (nullable = true)
 |-- MemSectorType: string (nullable = true)
 |-- YoJ: integer (nullable = true)
 |-- q_members_t: double (nullable = true)
 |-- q_leavers_t: double (nullable = true)

Sample YoB and YoJ values (post-cast):
----------------------------------------------------------------------
+----+----+
|YoB |YoJ |
+----+----+
|1993|2017|
|1990|2020|
|1989|2020|
|1990|2020|
|1990|2020|
+----+----+
only showing top 5 rows


**RESULT**

`YoB` and `YoJ` have been successfully cast from `double` to `integer` type. The schema update is confirmed with no value distortion and sample values show clean four-digit year integers with no floating point artefacts. All remaining columns retain their original types as expected.

**Status:** ✓ Pass

### 2.2 YoB Outlier Remediation

**CONTEXT**

Notebook 01 identified 18 records where `YoB` contains impossible values, comprising 17 future year records (2020 to 2966) and 1 pre-1900 record. These represent clear data entry errors and cannot be corrected with certainty without access to the Organisation's source membership system. The single record at `YoB = 2966` is likely a century transposition error, however imputation without source verification would introduce false precision into the dataset.

**PURPOSE**

To ensure:
1. All impossible `YoB` values are identified and remediated
2. Affected rows are retained with their membership counts preserved
3. Corrections are transparent and auditable
4. No false precision is introduced through unsupported imputation

**STEP**

Identify all records where `YoB` falls outside the valid range of 1895 to 2007. Apply a tiered remediation strategy: confirmed errors (`YoB > 2007` or `YoB < 1895`) are nulled and flagged as `YOB_CONFIRMED_ERROR` or `YOB_PRE_1895`; records with `YoB = 2008` are retained and flagged as `YOB_YOUNG_MEMBER` as these represent plausible young members on vocational nursing pathways. Validate that no confirmed errors remain after remediation.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.2 YoB Outlier Remediation
# ═══════════════════════════════════════════════════════════════

# Profile outliers before remediation
print("YoB Outliers (pre-remediation):")
print("-" * 70)
df.filter((F.col("YoB") < YOB_MIN) | (F.col("YoB") > YOB_MAX)) \
  .select("_c0", "CM_snapshot_date", "Region", "Branch", "YoB", "YoJ", "q_members_t") \
  .orderBy(F.col("YoB").desc()) \
  .show(25, truncate=False)

# Capture counts for reporting
yob_confirmed_errors = df.filter(F.col("YoB") > YOB_MAX).count()
yob_young_members = df.filter(F.col("YoB") == 2008).count()

print(f"Confirmed errors (YoB > {YOB_MAX})  : {yob_confirmed_errors}")
print(f"Young member flags (YoB = 2008)     : {yob_young_members}")
print("-" * 70)

# Apply tiered remediation
df = df.withColumn(
    "cleaning_flag",
    F.when(F.col("YoB") == 2008, F.lit("YOB_YOUNG_MEMBER"))
     .when(F.col("YoB") > YOB_MAX, F.lit("YOB_CONFIRMED_ERROR"))
     .when(F.col("YoB") < YOB_MIN, F.lit("YOB_PRE_1895"))
     .otherwise(F.lit(None).cast(StringType()))
).withColumn(
    "YoB",
    F.when(F.col("YoB") == 2008, F.col("YoB"))
     .when(F.col("YoB") > YOB_MAX, F.lit(None).cast(IntegerType()))
     .when(F.col("YoB") < YOB_MIN, F.lit(None).cast(IntegerType()))
     .otherwise(F.col("YoB"))
)

# Validate remediation
remaining_errors = df.filter(
    (F.col("YoB") > YOB_MAX) & 
    (F.col("cleaning_flag") != "YOB_YOUNG_MEMBER")
).count()

print(f"Confirmed errors remaining  : {remaining_errors}  (expected: 0)")
print(f"Young member records flagged: {df.filter(F.col('cleaning_flag') == 'YOB_YOUNG_MEMBER').count()}")
print(f"Total cleaning flags applied: {df.filter(F.col('cleaning_flag').isNotNull()).count()}")

YoB Outliers (pre-remediation):
----------------------------------------------------------------------
+--------+----------------+----------------------+----------------------------+----+----+-----------------+
|_c0     |CM_snapshot_date|Region                |Branch                      |YoB |YoJ |q_members_t      |
+--------+----------------+----------------------+----------------------------+----+----+-----------------+
|11829974|2024-04-01      |Northern              |Northumberland Tyne and Wear|2966|2024|2.969121140142518|
|8911643 |2023-07-01      |H Q (Overseas)        |Overseas Branch             |2044|1985|2.969121140142518|
|8337560 |2023-05-01      |South West            |Dorset                      |2042|2023|2.969121140142518|
|7068053 |2023-01-01      |North West            |Greater Manchester          |2029|1969|2.969121140142518|
|7677183 |2023-03-01      |North West            |Greater Manchester          |2029|1969|2.969121140142518|
|8297098 |2023-05-01      |North 

**RESULT**

111 records were identified with `YoB` values outside the valid range of 1895 to 2007. A tiered remediation strategy was applied: 17 confirmed data entry errors with impossible future birth years (2020 to 2966) were nulled and flagged as `YOB_CONFIRMED_ERROR`; 94 records with `YoB = 2008` were retained with a `YOB_YOUNG_MEMBER` flag, representing plausible young members on vocational nursing pathways with a join age of 16 to 17. The higher record count relative to Notebook 01's 18 distinct outliers reflects the panel structure of the dataset, where a single erroneous birth year persists across multiple monthly snapshots for the same member. All confirmed errors have been remediated and no out-of-range values remain.

**Status:** ✓ Pass

### 2.3 Age Violation Resolution

**CONTEXT**

Notebook 01 identified 1,183 records where the derived join age (`YoJ - YoB`) falls outside the plausible range of 14 to 90 years. These violations include negative join ages, implausibly young join ages, and implausibly old join ages. Before applying any remediation, a structured profiling exercise is conducted to identify whether violations follow a systematic pattern, ensuring the remediation strategy is evidence-based rather than assumption-driven.

**PURPOSE**

To ensure:
1. Age violations are profiled and understood before any remediation is applied
2. Systematic patterns are identified and treated distinctly from isolated data entry errors
3. Affected `YoB` values are nulled using a tiered flag strategy that preserves analytical transparency
4. Row membership counts are preserved, no records are dropped
5. All remediation decisions are documented for auditability and downstream exclusion

**STEP**

Profile all records where the derived join age falls outside the valid range of 14 to 90 years, classifying violations by sub-type. Conduct a deep dive into `TOO_OLD` records to identify systematic patterns prior to remediation. Apply a tiered remediation strategy based on profiling findings, null affected `YoB` values, apply cleaning flags by sub-type, and validate that no violations remain.

#### 2.3.1 Age Violation Profiling

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3.1 Age Violation Profiling
# ═══════════════════════════════════════════════════════════════

# Calculate join age and profile violations
print("Age Violations (pre-remediation):")
print("-" * 70)

age_violations_df = df.filter(
    F.col("YoB").isNotNull() & F.col("YoJ").isNotNull()
).withColumn(
    "join_age", F.col("YoJ") - F.col("YoB")
).filter(
    (F.col("join_age") < MIN_JOIN_AGE) | (F.col("join_age") > MAX_JOIN_AGE)
).withColumn(
    "violation_type",
    F.when(F.col("join_age") < 0, F.lit("NEGATIVE_AGE"))
     .when(F.col("join_age") < MIN_JOIN_AGE, F.lit("TOO_YOUNG"))
     .otherwise(F.lit("TOO_OLD"))
)

# Breakdown by violation type
age_violations_df.groupBy("violation_type") \
    .agg(
        F.count("*").alias("record_count"),
        F.min("join_age").alias("min_age"),
        F.max("join_age").alias("max_age")
    ).orderBy("violation_type").show(truncate=False)

age_violation_count = age_violations_df.count()
print(f"Total age violations identified: {age_violation_count}")

Age Violations (pre-remediation):
----------------------------------------------------------------------
+--------------+------------+-------+-------+
|violation_type|record_count|min_age|max_age|
+--------------+------------+-------+-------+
|NEGATIVE_AGE  |22          |-4     |-2     |
|TOO_OLD       |119684      |91     |126    |
|TOO_YOUNG     |900         |0      |13     |
+--------------+------------+-------+-------+

Total age violations identified: 120606


**RESULT**

120,606 age violations were identified across three sub-types. `TOO_OLD` dominates with 119,684 records, representing join ages of 91 to 126 years, strongly suggesting a systematic issue rather than isolated data entry errors. `TOO_YOUNG` accounts for 900 records with join ages of 0 to 13 years, and `NEGATIVE_AGE` captures 22 records where `YoJ` precedes `YoB` by 2 to 4 years. The volume of `TOO_OLD` records warrants a dedicated deep dive before remediation is applied.

**Status:** ⚠️ Investigate

#### 2.3.2 TOO_OLD Deep Dive

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3.2 TOO_OLD Deep Dive
# ═══════════════════════════════════════════════════════════════

too_old_pre_df = df.filter(
    F.col("YoB").isNotNull() & F.col("YoJ").isNotNull()
).withColumn(
    "join_age", F.col("YoJ") - F.col("YoB")
).filter(F.col("join_age") > MAX_JOIN_AGE)

# YoB distribution for TOO_OLD records
print("YoB Distribution (TOO_OLD records):")
print("-" * 70)
too_old_pre_df.groupBy("YoB") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("YoB") \
    .show(50, truncate=False)

# YoJ vs YoB cross tab
print("YoJ vs YoB Sample (TOO_OLD records):")
print("-" * 70)
too_old_pre_df.select("YoB", "YoJ", "join_age") \
    .groupBy("YoB", "YoJ", "join_age") \
    .count() \
    .orderBy("join_age", ascending=False) \
    .show(50, truncate=False)

YoB Distribution (TOO_OLD records):
----------------------------------------------------------------------
+----+------------+
|YoB |record_count|
+----+------------+
|1895|1           |
|1900|119579      |
|1908|1           |
|1920|10          |
|1921|1           |
|1922|8           |
|1924|1           |
|1928|16          |
|1930|9           |
|1931|15          |
|1933|34          |
|1934|9           |
+----+------------+

YoJ vs YoB Sample (TOO_OLD records):
----------------------------------------------------------------------
+----+----+--------+-----+
|YoB |YoJ |join_age|count|
+----+----+--------+-----+
|1895|2021|126     |1    |
|1900|2025|125     |145  |
|1900|2024|124     |381  |
|1900|2023|123     |787  |
|1900|2022|122     |889  |
|1900|2021|121     |807  |
|1900|2020|120     |1069 |
|1900|2019|119     |2173 |
|1900|2018|118     |2342 |
|1900|2017|117     |1970 |
|1900|2016|116     |879  |
|1900|2015|115     |650  |
|1908|2022|114     |1    |
|1900|2014|114     |1029 |
|1900

**RESULT**

The deep dive confirms a clear systematic pattern. `YoB = 1900` accounts for 119,579 of the 119,684 `TOO_OLD` records, spanning join years from 1991 to 2025 with entirely plausible `YoJ` values. This is a classic legacy system default placeholder, used when a member's birth year was unknown or not captured at the point of registration, rather than a genuine birth year. The remaining 105 records carry birth years ranging from 1895 to 1934, representing a smaller group of potentially genuine elderly or lifetime members and isolated data entry errors. These two distinct groups require separate treatment in the remediation strategy applied in 2.3.3.

**Status:** ⚠️ Investigate

#### 2.3.3 Age Violation Remediation

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.3.3 Age Violation Remediation
# ═══════════════════════════════════════════════════════════════

# Apply tiered remediation strategy
df = df.withColumn(
    "join_age_check",
    F.when(F.col("YoB").isNotNull() & F.col("YoJ").isNotNull(),
           F.col("YoJ") - F.col("YoB"))
     .otherwise(F.lit(None).cast(IntegerType()))
).withColumn(
    "cleaning_flag",
    F.coalesce(
        F.col("cleaning_flag"),
        F.when(F.col("YoB") == 1900, F.lit("YOB_DEFAULT_PLACEHOLDER"))
         .when(F.col("join_age_check") < 0, F.lit("AGE_NEGATIVE"))
         .when(F.col("join_age_check") < MIN_JOIN_AGE, F.lit("AGE_TOO_YOUNG"))
         .when((F.col("join_age_check") > MAX_JOIN_AGE) & 
               (F.col("YoB") <= 1934), F.lit("YOB_ELDERLY_MEMBER"))
         .when(F.col("join_age_check") > MAX_JOIN_AGE, F.lit("AGE_TOO_OLD"))
    )
).withColumn(
    "YoB",
    F.when(F.col("YoB") == 1900, F.lit(None).cast(IntegerType()))
     .when(F.col("join_age_check") < 0, F.lit(None).cast(IntegerType()))
     .when(F.col("join_age_check") < MIN_JOIN_AGE, F.lit(None).cast(IntegerType()))
     .otherwise(F.col("YoB"))
).drop("join_age_check")

# Validate remediation - exclude deliberately flagged edge cases
remaining_violations = df.filter(
    F.col("YoB").isNotNull() & F.col("YoJ").isNotNull()
).withColumn(
    "join_age", F.col("YoJ") - F.col("YoB")
).filter(
    ((F.col("join_age") < MIN_JOIN_AGE) | (F.col("join_age") > MAX_JOIN_AGE)) &
    (~F.col("cleaning_flag").isin("YOB_YOUNG_MEMBER", "YOB_ELDERLY_MEMBER"))
).count()

# Cleaning flag summary
print("Cleaning Flag Summary (post-remediation):")
print("-" * 70)
df.groupBy("cleaning_flag") \
  .count() \
  .orderBy(F.col("count").desc()) \
  .show(truncate=False)

print(f"Age violations remaining: {remaining_violations}  (expected: 0)")

Cleaning Flag Summary (post-remediation):
----------------------------------------------------------------------
+-----------------------+--------+
|cleaning_flag          |count   |
+-----------------------+--------+
|NULL                   |18190281|
|YOB_DEFAULT_PLACEHOLDER|270062  |
|AGE_TOO_YOUNG          |899     |
|YOB_ELDERLY_MEMBER     |105     |
|YOB_YOUNG_MEMBER       |94      |
|AGE_NEGATIVE           |22      |
|YOB_CONFIRMED_ERROR    |17      |
+-----------------------+--------+

Age violations remaining: 0  (expected: 0)


**RESULT**

A tiered remediation strategy was applied to 120,606 age violations, producing six distinct cleaning flags that preserve analytical transparency. The dominant finding was `YoB = 1900` as a legacy system default placeholder, accounting for 270,062 records across all 60 snapshots where a member's birth year was unknown at the point of registration. These records carry entirely plausible `YoJ` values confirming they represent legitimate members with missing birth year data rather than errors.

The remaining violations were handled as follows: 899 records with join ages of 0 to 13 were flagged as `AGE_TOO_YOUNG`; 22 records with negative join ages were flagged as `AGE_NEGATIVE`; 17 confirmed future year errors from Section 2.2 retain their `YOB_CONFIRMED_ERROR` flag. The 105 `YOB_ELDERLY_MEMBER` records with birth years ranging from 1895 to 1934 were retained with their `YoB` values preserved. While join ages of 91 to 126 are implausible for new registrations, two possibilities cannot be dismissed: these may represent genuine lifetime or honorary members whose records contain administrative data entry errors, or they may reflect deceased members who remain on the system pending administrative removal, a known occurrence in large membership organisations. The single `YoB = 1895` record with a join year of 2021 is the most anomalous, as joining at age 126 is not defensible under either hypothesis, however without access to the Organisation's source membership system no correction can be made with certainty. All affected rows are retained with cleaning flags applied for downstream exclusion if required.

**Status:** ✓ Pass

### 2.4 MemSectorType Standardisation

**CONTEXT**

Notebook 01 identified 307,221 records with inconsistent casing in `MemSectorType`, where the same employment sector label appears in multiple forms such as `"nhs"`, `"NHS"`, and `"Nhs"`. This is a systematic formatting inconsistency rather than a data entry error, likely resulting from data being sourced from multiple systems or entry points over the five-year observation period.

**PURPOSE**

To ensure:
1. All `MemSectorType` values are standardised to a consistent format
2. Group-by aggregations return the correct number of distinct categories
3. Power BI dashboard visuals display clean, consistent sector labels
4. Standardisation is applied without loss of any records or membership counts

**STEP**

Profile all distinct `MemSectorType` values and their record counts before standardisation. Apply `initcap()` and `trim()` to normalise casing and remove whitespace. Validate that the resulting distinct value set matches the expected category taxonomy.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.4 MemSectorType Standardisation
# ═══════════════════════════════════════════════════════════════

# Profile distinct values before standardisation
print("MemSectorType Distinct Values (pre-standardisation):")
print("-" * 70)
df.groupBy("MemSectorType") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(30, truncate=False)

# Apply standardisation
df = df.withColumn(
    "MemSectorType",
    F.when(F.col("MemSectorType").isNotNull(),
           F.initcap(F.trim(F.col("MemSectorType"))))
     .otherwise(F.lit(None).cast(StringType()))
)

# Override NHS back to all-caps after initcap
df = df.withColumn(
    "MemSectorType",
    F.when(F.col("MemSectorType") == "Nhs", F.lit("NHS"))
     .otherwise(F.col("MemSectorType"))
)

# Validate post-standardisation
print("MemSectorType Distinct Values (post-standardisation):")
print("-" * 70)
df.groupBy("MemSectorType") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(20, truncate=False)

distinct_count = df.select("MemSectorType").distinct().count()
print(f"Distinct MemSectorType values post-standardisation: {distinct_count}")

MemSectorType Distinct Values (pre-standardisation):
----------------------------------------------------------------------
+-------------------+------------+--------------------+
|MemSectorType      |record_count|total_members       |
+-------------------+------------+--------------------+
|NHS                |10024494    |6.153519893130582E7 |
|Independent        |4981521     |1.938356888370331E7 |
|Education          |1711549     |1.0590148456092356E7|
|NULL               |1436695     |4468711.401420915   |
|Other public sector|267728      |843144.2992871267   |
|Other Public Sector|39493       |119242.87410926285  |
+-------------------+------------+--------------------+

MemSectorType Distinct Values (post-standardisation):
----------------------------------------------------------------------
+-------------------+------------+--------------------+
|MemSectorType      |record_count|total_members       |
+-------------------+------------+--------------------+
|NHS                |1

**RESULT**

`MemSectorType` has been successfully standardised across all records. The pre-standardisation profile revealed two casing variants of `Other Public Sector`, accounting for 307,221 records, which have been consolidated into a single canonical label. `NHS` was preserved in its correct all-caps form via an explicit override following `initcap()` normalisation, reflecting its status as a proper acronym in the UK health sector. The final distinct value set comprises five categories: `NHS`, `Independent`, `Education`, `Other Public Sector`, and `null`. Null values are retained and will be investigated in Section 2.5. All 18,461,480 rows are preserved with no membership counts affected.

**Status:** ✓ Pass

### 2.5 MemSectorType Null Investigation

**CONTEXT**

Following standardisation in Section 2.4, `MemSectorType` retains 1,436,695 null values representing 7.78% of all records. Notebook 01 flagged this as a low severity issue requiring investigation. Unlike the casing inconsistency which was a formatting error, these nulls may represent legitimate missing data, a specific membership category that does not map to a sector, or a systematic gap in data collection over a defined period.

**PURPOSE**

To ensure:
1. The null pattern in `MemSectorType` is understood before any treatment is applied
2. Nulls are classified as either legitimate missing data or a systematic collection gap
3. An informed decision is made on whether to impute, flag, or retain the null values
4. Downstream segmentation analysis correctly accounts for the null population

**STEP**

Profile the null `MemSectorType` records across time, region, membership category, and branch to identify whether the null pattern is random or concentrated in specific segments. Cross-reference against `MemCategory` and `CatName` to determine if a particular membership type is systematically missing sector data. Conduct a membership hierarchy cross-tab to quantify the proportion of nulls attributable to non-active employment categories versus ambiguous categories.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.5 MemSectorType Null Investigation
# ═══════════════════════════════════════════════════════════════

null_sector_df = df.filter(F.col("MemSectorType").isNull())

# Temporal distribution of nulls
print("Null MemSectorType by Snapshot Date:")
print("-" * 70)
null_sector_df.groupBy("CM_snapshot_date") \
    .agg(F.count("*").alias("null_count"),
         F.sum("q_members_t").alias("total_members")) \
    .orderBy("CM_snapshot_date") \
    .show(60, truncate=False)

# MemCategory distribution of nulls
print("Null MemSectorType by MemCategory:")
print("-" * 70)
null_sector_df.groupBy("MemCategory") \
    .agg(F.count("*").alias("null_count"),
         F.sum("q_members_t").alias("total_members")) \
    .orderBy(F.col("null_count").desc()) \
    .show(truncate=False)

# CatName distribution of nulls
print("Null MemSectorType by CatName:")
print("-" * 70)
null_sector_df.groupBy("CatName") \
    .agg(F.count("*").alias("null_count"),
         F.sum("q_members_t").alias("total_members")) \
    .orderBy(F.col("null_count").desc()) \
    .show(truncate=False)

# Region distribution of nulls
print("Null MemSectorType by Region:")
print("-" * 70)
null_sector_df.groupBy("Region") \
    .agg(F.count("*").alias("null_count"),
         F.sum("q_members_t").alias("total_members")) \
    .orderBy(F.col("null_count").desc()) \
    .show(truncate=False)

Null MemSectorType by Snapshot Date:
----------------------------------------------------------------------
+----------------+----------+-----------------+
|CM_snapshot_date|null_count|total_members    |
+----------------+----------+-----------------+
|2021-01-01      |24582     |76633.01662708473|
|2021-02-01      |24564     |76517.2209026193 |
|2021-03-01      |24532     |76359.85748219198|
|2021-04-01      |24575     |76472.6840855229 |
|2021-05-01      |24474     |76125.29691212092|
|2021-06-01      |24420     |75976.84085511418|
|2021-07-01      |24418     |75950.11876485284|
|2021-08-01      |24288     |75513.65795726675|
|2021-09-01      |23923     |74361.6389548792 |
|2021-10-01      |23942     |74400.2375297011 |
|2021-11-01      |24012     |74616.98337293115|
|2021-12-01      |23819     |74011.28266035124|
|2022-01-01      |23340     |72541.56769597503|
|2022-02-01      |22923     |71250.00000001528|
|2022-03-01      |22911     |71232.1852731744 |
|2022-04-01      |23049     

#### 2.5.1 Membership Hierarchy Cross-Tab

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.5.1 MemSectorType Null - Hierarchy Cross-Tab
# ═══════════════════════════════════════════════════════════════

# Define non-active employment CatName categories
non_active_categories = [
    "Nurse - Retired",
    "Nursing Support Worker - Retired",
    "Nurse - Career Break",
    "Nurse - Voluntary Break",
    "Nursing Support Worker -Career Break",
    "Nursing Support Worker - Voluntary Break"
]

# Cross-tab: null MemSectorType by active vs non-active employment status
null_sector_df = df.filter(F.col("MemSectorType").isNull())

null_sector_df.withColumn(
    "employment_status",
    F.when(F.col("CatName").isin(non_active_categories), F.lit("Non-Active Employment"))
     .otherwise(F.lit("Active Employment"))
).groupBy("employment_status") \
 .agg(F.count("*").alias("null_count"),
      F.sum("q_members_t").alias("total_members")) \
 .withColumn("pct_of_nulls",
             F.round(F.col("null_count") / 1436695 * 100, 2)) \
 .orderBy(F.col("null_count").desc()) \
 .show(truncate=False)

+---------------------+----------+------------------+------------+
|employment_status    |null_count|total_members     |pct_of_nulls|
+---------------------+----------+------------------+------------+
|Non-Active Employment|762888    |2400970.9026097716|53.1        |
|Active Employment    |673807    |2067740.4988098382|46.9        |
+---------------------+----------+------------------+------------+



**RESULT**

`MemSectorType` nulls are consistent across all 60 snapshots, ranging from approximately 22,900 to 24,900 records per month, confirming this is a structural characteristic of the dataset rather than a data collection gap at a specific point in time. Regional distribution is broadly proportional to region size with no single region standing out as anomalous.

Cross-referencing against the membership hierarchy reveals that the null pattern is driven primarily by membership category rather than data collection failure. `Nurse - Retired` accounts for 746,956 null records, the single largest contributor. The hierarchy cross-tab confirms that 53.1% of null `MemSectorType` records belong to definitively non-active employment categories where sector data is structurally not applicable by definition. The remaining 46.9% falls across ambiguous categories including `Nurse Full`, `Student`, and `Life Member`. It is worth noting that `Student` as a category provides no reliable indication of employment status, as student nurses may be on placement and earning or purely academic depending on their stage of training. `Life Member` requires further investigation in Section 2.5.2 to determine whether it represents a non-active status or a membership tier before a definitive classification can be assigned.

**Status:** ⚠️ Investigate

#### 2.5.2 Life Member Sector Verification

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.5.2 Life Member Sector Verification
# ═══════════════════════════════════════════════════════════════

# Cross-tab Life Member records against MemSectorType
print("Life Member MemSectorType Distribution:")
print("-" * 70)
df.filter(F.col("CatName") == "Life Member") \
  .groupBy("MemSectorType") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(truncate=False)

Life Member MemSectorType Distribution:
----------------------------------------------------------------------
+-------------------+------------+-----------------+
|MemSectorType      |record_count|total_members    |
+-------------------+------------+-----------------+
|NULL               |30309       |92036.81710213267|
|NHS                |305         |905.5819477434676|
|Independent        |225         |668.0522565320667|
|Other Public Sector|25          |74.22802850356295|
+-------------------+------------+-----------------+



**RESULT**

555 `Life Member` records carry active employment sector values across NHS, Independent, and Other Public Sector, confirming that `Life Member` is a membership tier rather than an employment status classification. The 30,309 null `MemSectorType` records within this category therefore cannot be classified as structurally not applicable and should be treated as **Missing At Random**. This finding was resolved through direct data analysis, removing the need for stakeholder consultation.

### 2.6 MemCategory Label Deduplication

**CONTEXT**

Notebook 01 identified a naming change in `MemCategory` where `Nursing Support Worker` was systematically relabelled to `Nurse Support Worker` in May 2024, affecting 1,729,737 records. This was documented as a clean systematic transition rather than a data entry error. However the label change creates two distinct values representing the same membership category across the observation period, which would cause incorrect category splits in group-by aggregations and dashboard visuals if left unaddressed.

**PURPOSE**

To ensure:
1. `MemCategory` contains a consistent set of canonical labels across all 60 snapshots
2. The naming transition in May 2024 is resolved into a single canonical label
3. Group-by aggregations return the correct number of distinct categories
4. The transition is documented transparently for downstream analysis

**STEP**

Profile all distinct `MemCategory` values and their record counts. Apply whitespace normalisation via `trim()` and internal whitespace collapse. Map `Nursing Support Worker` to the canonical label `Nurse Support Worker` to align with the Organisation's current official terminology. Validate that the resulting distinct value set contains the expected three categories.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.6 MemCategory Label Deduplication
# ═══════════════════════════════════════════════════════════════

# Profile distinct values before deduplication
print("MemCategory Distinct Values (pre-deduplication):")
print("-" * 70)
df.groupBy("MemCategory") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(truncate=False)

# Apply whitespace normalisation
df = df.withColumn(
    "MemCategory",
    F.regexp_replace(F.trim(F.col("MemCategory")), r"\s+", " ")
)

# Apply canonical label mapping
# Nursing Support Worker → Nurse Support Worker (the organisation official rename May 2024)
df = df.withColumn(
    "MemCategory",
    F.when(F.col("MemCategory") == "Nursing Support Worker",
           F.lit("Nurse Support Worker"))
     .otherwise(F.col("MemCategory"))
)

# Validate post-deduplication
print("MemCategory Distinct Values (post-deduplication):")
print("-" * 70)
df.groupBy("MemCategory") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(truncate=False)

distinct_count = df.select("MemCategory").distinct().count()
print(f"Distinct MemCategory values post-deduplication: {distinct_count}  (expected: 3)")

MemCategory Distinct Values (pre-deduplication):
----------------------------------------------------------------------
+----------------------+------------+-------------------+
|MemCategory           |record_count|total_members      |
+----------------------+------------+-------------------+
|Nurse member          |16022515    |8.332830166236542E7|
|Nursing Support Worker|1054250     |4044625.8907405348 |
|Student               |709228      |6701713.182902     |
|Nurse Support Worker  |675487      |2865374.109266992  |
+----------------------+------------+-------------------+

MemCategory Distinct Values (post-deduplication):
----------------------------------------------------------------------
+--------------------+------------+-------------------+
|MemCategory         |record_count|total_members      |
+--------------------+------------+-------------------+
|Nurse member        |16022515    |8.332830166236542E7|
|Nurse Support Worker|1729737     |6910000.000019052  |
|Student      

**RESULT**

`MemCategory` has been successfully deduplicated from four labels to three canonical categories. `Nursing Support Worker` representing 1,054,250 records prior to May 2024 has been mapped to the current official label `Nurse Support Worker`, consolidating 1,729,737 records under a single canonical label. The final three categories are `Nurse member`, `Nurse Support Worker`, and `Student`, consistent with the Organisation's current membership taxonomy.

**Status:** ✓ Pass

### 2.7 Geographic Anomaly Handling

**CONTEXT**

Notebook 01 identified two geographic anomalies in the `Region` column: one null region record and one record carrying the value `"Non Members"` in a geographic field. Both were flagged as low severity with a combined impact of 2 records across the dataset. The null region represents a snapshot segment with no geographic assignment, while `"Non Members"` appears to be a category label misplaced in a geographic field. At 0.000011% of the dataset their analytical impact is negligible.

**PURPOSE**

To ensure:
1. Null region values are handled to prevent null exclusion errors in group-by aggregations
2. The `"Non Members"` anomaly is flagged and documented for transparency
3. No records are dropped, all membership counts are preserved
4. Downstream regional analysis and Power BI dashboard visuals are not affected by anomalous values

**STEP**

Profile all distinct `Region` values to confirm the two anomalies identified in Notebook 01. Replace null region values with `"Unknown"` to prevent silent exclusion in aggregations. Flag the `"Non Members"` record with a `geo_flag` column for transparency. Validate that no null region values remain after handling.

In [0]:
# ═══════════════════════════════════════════════════════════════
# 2.7 Geographic Anomaly Handling
# ═══════════════════════════════════════════════════════════════

# Profile Region distinct values before handling
print("Region Distinct Values (pre-handling):")
print("-" * 70)
df.groupBy("Region") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(20, truncate=False)

# Apply geographic anomaly handling
df = df.withColumn(
    "geo_flag",
    F.when(F.col("Region").isNull(), F.lit("REGION_NULL"))
     .when(F.col("Region") == "Non Members", F.lit("REGION_NON_MEMBERS"))
     .otherwise(F.lit(None).cast(StringType()))
).withColumn(
    "Region",
    F.when(F.col("Region").isNull(), F.lit("Unknown"))
     .otherwise(F.col("Region"))
)

# Validate handling
null_remaining = df.filter(F.col("Region").isNull()).count()
geo_flagged = df.filter(F.col("geo_flag").isNotNull()).count()

print("Region Distinct Values (post-handling):")
print("-" * 70)
df.groupBy("Region") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(20, truncate=False)

print(f"Null regions remaining : {null_remaining}  (expected: 0)")
print(f"Geo flagged records    : {geo_flagged}")

Region Distinct Values (pre-handling):
----------------------------------------------------------------------
+----------------------+------------+--------------------+
|Region                |record_count|total_members       |
+----------------------+------------+--------------------+
|South East            |2675078     |1.2304331947810939E7|
|London                |2406024     |1.2727609857540948E7|
|North West            |1804964     |1.1079462589108424E7|
|West Midlands         |1775766     |8677084.323066626   |
|South West            |1741816     |8423669.83375392    |
|Scotland              |1607809     |8803833.135415303   |
|Eastern               |1524718     |7913266.0332727     |
|East Midlands         |1277500     |6606540.973881954   |
|Yorkshire & The Humber|1142390     |7509124.109274897   |
|Wales                 |1067920     |5482185.273161985   |
|Northern              |728556      |4084818.8836068376  |
|Northern Ireland      |684325      |3248990.498809264   |
|H Q 

**RESULT**

Two geographic anomalies were confirmed and handled. The single null `Region` record has been replaced with `"Unknown"` to prevent silent exclusion in downstream group-by aggregations, and the single `"Non Members"` record has been retained with a `geo_flag` of `REGION_NON_MEMBERS`. At 0.000005% of the dataset this anomaly has no analytical impact and the `geo_flag` provides sufficient documentation for transparency. All 18,461,480 rows are preserved with no membership counts affected. The 13 legitimate geographic regions plus `H Q (Overseas)` remain intact and unmodified.

It is worth noting that `Int_nurse` is not part of the `Region` → `Branch` geographic hierarchy. It classifies members by their international nursing registration origin, comprising `UK`, `EEU`, `Overseas`, and `Other`, and sits as an independent registration classification dimension alongside the geographic hierarchy rather than within it.

**Status:** ✓ Pass

## 3.0 Summary & Conclusions

### 3.1 Cleaning Summary

In [0]:
# ═══════════════════════════════════════════════════════════════
# 3.1 Cleaning Summary
# ═══════════════════════════════════════════════════════════════

print("Full Cleaning Flag Summary (post all transformations):")
print("-" * 70)
df.groupBy("cleaning_flag") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .orderBy(F.col("record_count").desc()) \
  .show(truncate=False)

print("Geo Flag Summary:")
print("-" * 70)
df.filter(F.col("geo_flag").isNotNull()) \
  .groupBy("geo_flag") \
  .agg(F.count("*").alias("record_count"),
       F.sum("q_members_t").alias("total_members")) \
  .show(truncate=False)

print(f"Total flagged records  : {df.filter(F.col('cleaning_flag').isNotNull()).count():,}")
print(f"Total clean records    : {df.filter(F.col('cleaning_flag').isNull()).count():,}")
print(f"Total records          : {df.count():,}")

Full Cleaning Flag Summary (post all transformations):
----------------------------------------------------------------------
+-----------------------+------------+-------------------+
|cleaning_flag          |record_count|total_members      |
+-----------------------+------------+-------------------+
|NULL                   |18190281    |9.579755344299307E7|
|YOB_DEFAULT_PLACEHOLDER|270062      |1138904.3942988615 |
|AGE_TOO_YOUNG          |899         |2669.2399049881224 |
|YOB_ELDERLY_MEMBER     |105         |311.7577197149644  |
|YOB_YOUNG_MEMBER       |94          |460.2137767220903  |
|AGE_NEGATIVE           |22          |65.3206650831354   |
|YOB_CONFIRMED_ERROR    |17          |50.47505938242281  |
+-----------------------+------------+-------------------+

Geo Flag Summary:
----------------------------------------------------------------------
+------------------+------------+-----------------+
|geo_flag          |record_count|total_members    |
+------------------+-----------

### 3.2 Data Quality Assessment

Notebook 02 addressed all data quality issues identified in Notebook 01 across seven cleaning and transformation subsections. A total of 271,199 records (1.47% of the dataset) carry a cleaning flag, with the remaining 18,190,281 records (98.53%) confirmed clean and analysis-ready.

The dominant finding was `YOB_DEFAULT_PLACEHOLDER`, accounting for 270,062 records where `YoB = 1900` was used as a legacy system default for members whose birth year was unknown at the point of registration. This single pattern accounts for 99.58% of all flagged records and represents a structural characteristic of the dataset rather than a data quality failure.

The tiered remediation strategy applied across Sections 2.2 and 2.3 ensures that all flagged records are retained with their membership counts preserved. No records were dropped at any stage. Cleaning flags provide full auditability and allow downstream analysis to include or exclude specific flag types as required.

`MemSectorType` casing inconsistency affecting 307,221 records was fully resolved via `initcap()` normalisation with an explicit `NHS` override. `MemCategory` label deduplication successfully consolidated `Nursing Support Worker` into the current canonical label `Nurse Support Worker`, reflecting the Organisation's official May 2024 category rename. Both transformations are lossless with no impact on membership counts.

The `MemSectorType` null investigation confirmed that 53.1% of null records are structurally expected, belonging to definitively non-active employment categories where sector data is not applicable by definition. The remaining 46.9% sits across ambiguous categories including `Nurse Full` and `Student`. A targeted investigation of `Life Member` records confirmed that 555 life members carry active employment sector values, establishing that `Life Member` is a membership tier rather than an employment status. Null `MemSectorType` values within this category are therefore classified as **Missing At Random** and will be carried forward into the Notebook 03 missingness analysis accordingly.

Two geographic anomalies affecting a combined 2 records were handled via flagging, with null region replaced by `"Unknown"` and `"Non Members"` documented via `geo_flag`. Their combined analytical impact is negligible at 0.000011% of the dataset.

### 3.3 Silver Layer Creation

In [0]:
# ═══════════════════════════════════════════════════════════════
# 3.3 Silver Layer Creation
# ═══════════════════════════════════════════════════════════════

# Create Silver volume if not exists
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.rcn_churn.silver")
print("Silver volume confirmed")
print("-" * 70)

print(f"Writing Silver table to: {SILVER_PATH}")
print("-" * 70)

# Write cleaned DataFrame to Silver Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("CM_snapshot_date") \
  .option("overwriteSchema", "true") \
  .save(SILVER_PATH)

print("Silver table written successfully")
print("-" * 70)

# Verify Silver table
silver_df = spark.read.format("delta").load(SILVER_PATH)
silver_row_count = silver_df.count()
silver_col_count = len(silver_df.columns)

print(f"Rows    : {silver_row_count:,}  (expected: 18,461,480)")
print(f"Columns : {silver_col_count}")

assert silver_row_count == 18_461_480, \
    f"Row count mismatch: {silver_row_count:,}"

print("-" * 70)
print("Silver table verified and ready for downstream analysis")

Silver volume confirmed
----------------------------------------------------------------------
Writing Silver table to: /Volumes/workspace/rcn_churn/silver/churn_cleaned/
----------------------------------------------------------------------
Silver table written successfully
----------------------------------------------------------------------
Rows    : 18,461,480  (expected: 18,461,480)
Columns : 14
----------------------------------------------------------------------
Silver table verified and ready for downstream analysis


In [0]:
# ═══════════════════════════════════════════════════════════════
# Silver Table Verification
# ═══════════════════════════════════════════════════════════════

# Confirm volume exists
display(dbutils.fs.ls("/Volumes/workspace/rcn_churn/silver/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/rcn_churn/silver/churn_cleaned/,churn_cleaned/,0,1772409966065


### 3.4 Next Steps

Notebook 03 will conduct an exploratory deep dive into the cleaned Silver table, building on the validated foundation established in Notebooks 01 and 02. The following analytical workstreams are planned:

**Correlation Analysis**
Examine relationships between membership category, sector type, region, age, tenure, and churn behaviour to identify the strongest predictors of member attrition.

**Cohort Analysis & Retention Patterns**
Construct retention curves by year of join cohort to understand how long members typically remain active and at what tenure points churn risk is highest.

**Churn Rate Calculations**
Derive monthly and annual churn rates at segment level using `SUM(q_members_t)` and `SUM(q_leavers_t)`, producing the core metric that underpins all subsequent analysis.

**Member Segmentation Profiling**
Profile membership composition across region, sector, category, age band, and tenure band to establish the demographic and structural characteristics of the membership base.

**Temporal Trend Analysis**
Examine how membership size, composition, and churn rates have evolved across the 60-month observation period from January 2021 to December 2025.